<div style="background:linear-gradient(135deg,#4c0519 0%,#be123c 55%,#fb7185 100%);border-radius:18px;padding:32px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fecdd3;font-weight:700;text-transform:uppercase">Chapter 157 &middot; Communicating Results &middot; Report 3 of 3</div>
  <div style="font-size:32px;font-weight:900;line-height:1.1;margin:10px 0 6px">Analytical Report: Drivers of Customer Satisfaction</div>
  <div style="font-size:15px;color:#ffe4e6;max-width:760px;line-height:1.6">The full study. A longer analytical report needs a summary, a method, results with proper tables and figures, an interpretation, and stated limitations. This notebook fits a regression and writes the whole thing to Word.</div>
</div>

In [ ]:
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
# A clean house style for report-ready figures: no chartjunk, strong titles, muted grid.
mpl.rcParams.update({"figure.dpi":110,"font.size":11,"axes.spines.top":False,"axes.spines.right":False,
    "axes.grid":True,"grid.alpha":0.22,"axes.titleweight":"bold","axes.titlesize":12.5,
    "axes.titlelocation":"left","axes.titlepad":10})
ROSE, INK, MUT, GR, RD = "#be123c", "#1a2138", "#64748b", "#16a34a", "#dc2626"
BASE = "https://raw.githubusercontent.com/johnfisher-ai/Statistics-Data-Science-AI-Visual-Book/main/data/"
fn = "communicating-insights--company-data.xlsx"
def load(sheet):
    try: return pd.read_excel("../../data/" + fn, sheet_name=sheet)
    except FileNotFoundError: return pd.read_excel(BASE + fn, sheet_name=sheet)
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from pathlib import Path
import tempfile
ROSE_DOC, GREY_DOC = RGBColor(0xBE,0x12,0x3C), RGBColor(0x64,0x74,0x8B)
def new_report(title, subtitle):
    doc = Document()
    for s in doc.sections:                       # US Letter, sensible margins
        s.page_width, s.page_height = Inches(8.5), Inches(11)
        s.left_margin = s.right_margin = Inches(1); s.top_margin = s.bottom_margin = Inches(0.9)
    t = doc.add_heading(title, level=0)
    for r in t.runs: r.font.color.rgb = ROSE_DOC
    sp = doc.add_paragraph(subtitle); sp.runs[0].italic = True; sp.runs[0].font.color.rgb = GREY_DOC
    return doc
def h2(doc, text):
    hd = doc.add_heading(text, level=1)
    for r in hd.runs: r.font.color.rgb = ROSE_DOC
def bullets(doc, items):
    for it in items: doc.add_paragraph(it, style="List Bullet")
def df_table(doc, df):                           # df cells should already be display strings
    tb = doc.add_table(rows=1, cols=len(df.columns)); tb.style = "Light Grid Accent 1"
    for j, c in enumerate(df.columns):
        cell = tb.rows[0].cells[j]; cell.text = str(c)
        for r in cell.paragraphs[0].runs: r.bold = True
    for _, row in df.iterrows():
        cells = tb.add_row().cells
        for j, c in enumerate(df.columns): cells[j].text = str(row[c])
    return tb
def add_fig(doc, fig, width=6.3):
    p = Path(tempfile.mkdtemp()) / "fig.png"; fig.savefig(p, dpi=150, bbox_inches="tight"); plt.close(fig)
    doc.add_picture(str(p), width=Inches(width)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
def save_report(doc, name):                       # write into the book repo when present, else the cwd (Colab)
    outdir = Path("../../reports") if Path("../../reports").exists() else Path(".")
    outdir.mkdir(exist_ok=True); path = outdir / name; doc.save(path)
    print("wrote", path.resolve()); return path
import statsmodels.formula.api as smf

## Step 1 &middot; Fit the model
Regress overall satisfaction on four candidate drivers. The coefficients say how many points of satisfaction each one-point improvement in a driver buys, holding the others fixed.

In [ ]:
survey = load("Survey")
model = smf.ols("satisfaction ~ ease_of_use + support_quality + price_fairness + wait_time", survey).fit()
coefs = model.params.drop("Intercept").sort_values(ascending=False)
print(f"R-squared {model.rsquared:.3f}  | mean satisfaction {survey.satisfaction.mean():.1f}")
print(coefs.round(2).to_string())

## Step 2 &middot; Results figures
A coefficient plot ranks the drivers by impact, and a scatter shows the strongest one against satisfaction. Both carry titles that state the finding.

In [ ]:
fig_coef, ax = plt.subplots(figsize=(7.4, 3.0))
cvals = coefs.sort_values()
ax.barh(cvals.index.str.replace("_"," "), cvals.values, color=[RD if v<0 else ROSE for v in cvals.values])
ax.axvline(0, color=INK, lw=1); ax.set_title("Ease of use is the biggest driver; long waits hurt")
ax.set_xlabel("points of satisfaction per 1-point change in driver"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

fig_sc, ax = plt.subplots(figsize=(6.6, 3.2))
ax.scatter(survey.ease_of_use, survey.satisfaction, s=14, alpha=0.4, color=ROSE)
b0, b1 = np.polyfit(survey.ease_of_use, survey.satisfaction, 1)
xs = np.array([survey.ease_of_use.min(), survey.ease_of_use.max()])
ax.plot(xs, b0*xs+b1, color=INK, lw=2)
ax.set_title("Satisfaction rises steadily with ease of use"); ax.set_xlabel("ease of use (1-10)"); ax.set_ylabel("satisfaction (0-100)")
plt.tight_layout(); plt.show()

## Step 3 &middot; Write the analytical report
The long-form genre: summary, data and method, a real coefficient table, the figures, an interpretation, and limitations. This is the report a technical reviewer expects to be able to check line by line.

In [ ]:
doc = new_report("Drivers of Customer Satisfaction", "Analytics team  |  Survey of 320 customers")

h2(doc, "Summary")
top = coefs.index[0].replace("_"," ")
doc.add_paragraph(f"Across 320 respondents, four service factors explain {model.rsquared:.0%} of the variation in "
                  f"satisfaction. The strongest lever is {top} (each 1-point gain adds {coefs.iloc[0]:.1f} points of "
                  f"satisfaction), followed by support quality. Long wait times are the only factor that lowers it. "
                  f"Investment in {top} offers the largest expected return.")

h2(doc, "Data and method")
doc.add_paragraph("Respondents rated overall satisfaction from 0 to 100 and four drivers from 1 to 10. We fit an "
                  "ordinary least squares regression of satisfaction on all four drivers simultaneously, so each "
                  "coefficient is the effect of that driver holding the others constant.")

h2(doc, "Results")
tbl = pd.DataFrame({"Driver":[i.replace("_"," ") for i in model.params.index],
                    "Coefficient":[f"{v:+.2f}" for v in model.params.values],
                    "Std. error":[f"{v:.2f}" for v in model.bse.values],
                    "p-value":[f"{v:.3f}" for v in model.pvalues.values]})
df_table(doc, tbl)
add_fig(doc, fig_coef); add_fig(doc, fig_sc, width=5.6)

h2(doc, "Interpretation")
bullets(doc, [f"Ease of use and support quality together account for most of the explained variation.",
              f"Price fairness matters but less; wait time is the one factor to reduce, not increase.",
              f"The model explains {model.rsquared:.0%} of variation (R-squared = {model.rsquared:.2f}), a strong fit for survey data."])

h2(doc, "Limitations")
bullets(doc, ["The data is observational, so these are associations, not proven causes.",
              "Drivers are self-reported and may correlate with an overall halo effect.",
              "A single survey wave cannot capture how these relationships shift over time."])

path = save_report(doc, "report-3-satisfaction-analysis.docx")

### Wrap-up
The analytical genre earns trust by showing its work: a coefficient table with standard errors and p-values, figures that state findings, an interpretation grounded in the numbers, and honest limitations. Same discipline as the executive one-pager, more depth for a technical audience. Three genres, one habit: say the finding, show the evidence, own the uncertainty.